In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime


## Função Para Ler a Partição

In [0]:
def ler_ultima_particao_delta(spark, base_path):
  """
  Essa fução é para ler a ultima partição dos volumes delta baseada na coluna 'data_processamento'
  """
  try: 
      # Descobrir as partições direto no storage 
      particoes = dbutils.fs.ls(base_path)
      datas = [
              int(p.name.split('=')[1].replace('/', '')) 
              for p in particoes if "data_processamento=" in p.name
      ]
      
      if not datas:
          print(f"Nenhuma partição encontrada em {base_path}")
          return None
      else:
          ultima_particao = max(datas)
          print(f"[{base_path}] Ultima partição: {ultima_particao}")
          return spark.read.format("delta").load(f"{base_path}/data_processamento={ultima_particao}")
  except Exception as e:
    print(f"Erro ao ler caminho {base_path}: {e}")
    return None

## CVM - Fundos Imobiliarios - Geral

In [0]:
bronze_path_geral = "/Volumes/workspace/case_spark_cvm/bronze/cvm_fii_geral/"

df_silver_fii_geral = ler_ultima_particao_delta(spark, bronze_path_geral)

### 1.1 tratemento silver

#### 1.1.1 Normalizando CNPJ

In [0]:
df_silver_fii_geral = df_silver_fii_geral.withColumn(
    "CNPJ_FUNDO_CLASSE",
    f.regexp_replace(f.col("CNPJ_FUNDO_CLASSE"), "[^0-9]", "")
)

df_silver_fii_geral = df_silver_fii_geral.withColumn(
    "CNPJ_FUNDO_CLASSE",
    f.col("CNPJ_FUNDO_CLASSE").cast("long").cast("string")
)

#### 1.1.2 Retirando dados nulos de Colunas Cores

In [0]:
# Lista de colunas de caso falte dados precisamos dropa 
colunas_obrigatorias = ['Data_Referencia', 'CNPJ_FUNDO_CLASSE']

# Aplicando a filtro para dropar as colunas
df_bronze_cvm_ativo_passivo = df_bronze_cvm_ativo_passivo.dropna(subset=colunas_obrigatorias)

#### 1.1.3 Tratamento do Tipo de Dado

In [0]:
# Dropando a Data de Processamento da Bronze 
df_silver_fii_geral = df_silver_fii_geral.drop("data_processamento")

# Criando a Data de Processamento da silver
df_silver_fii_geral = df_silver_fii_geral.withColumn(
    "data_processamento",
    f.date_format(f.current_date(), "yyyyMMdd").cast("int")
)

In [0]:
df_silver_fii_geral = df_silver_fii_geral\
.withColumn('tipo_fundo_classe', f.col('Tipo_Fundo_Classe').cast(t.StringType())) \
.withColumn('cnpj_fundo_classe', f.col('CNPJ_FUNDO_CLASSE').cast(t.StringType())) \
.withColumn('data_referencia', f.col('Data_Referencia').cast(t.DateType())) \
.withColumn('versao', f.col('Versao').cast(t.IntegerType())) \
.withColumn('data_entrega', f.col('Data_Entrega').cast(t.DateType())) \
.withColumn('nome_fundo_classe', f.col('Nome_Fundo_Classe').cast(t.StringType())) \
.withColumn('data_funcionamento', f.col('Data_Funcionamento').cast(t.DateType())) \
.withColumn('publico_alvo', f.col('Publico_Alvo').cast(t.StringType())) \
.withColumn('codigo_isin', f.col('Codigo_ISIN').cast(t.StringType())) \
.withColumn('quantidade_cotas_emitidas', f.col('Quantidade_Cotas_Emitidas').cast(t.DecimalType(22, 2))) \
.withColumn('fundo_exclusivo', f.col('Fundo_Exclusivo').cast(t.StringType())) \
.withColumn('cotistas_vinculo_familiar', f.col('Cotistas_Vinculo_Familiar').cast(t.StringType())) \
.withColumn('mandato', f.col('Mandato').cast(t.StringType())) \
.withColumn('segmento_atuacao', f.col('Segmento_Atuacao').cast(t.StringType())) \
.withColumn('tipo_gestao', f.col('Tipo_Gestao').cast(t.StringType())) \
.withColumn('prazo_duracao', f.col('Prazo_Duracao').cast(t.StringType())) \
.withColumn('data_prazo_duracao', f.col('Data_Prazo_Duracao').cast(t.DateType())) \
.withColumn('encerramento_exercicio_social', f.col('Encerramento_Exercicio_Social').cast(t.StringType())) \
.withColumn('mercado_negociacao_bolsa', f.col('Mercado_Negociacao_Bolsa').cast(t.StringType())) \
.withColumn('mercado_negociacao_mbo', f.col('Mercado_Negociacao_MBO').cast(t.StringType())) \
.withColumn('mercado_negociacao_mb', f.col('Mercado_Negociacao_MB').cast(t.StringType())) \
.withColumn('entidade_administradora_bvmf', f.col('Entidade_Administradora_BVMF').cast(t.StringType())) \
.withColumn('entidade_administradora_cetip', f.col('Entidade_Administradora_CETIP').cast(t.StringType())) \
.withColumn('nome_administrador', f.col('Nome_Administrador').cast(t.StringType())) \
.withColumn('cnpj_administrador', f.col('CNPJ_Administrador').cast(t.StringType())) \
.withColumn('logradouro', f.col('Logradouro').cast(t.StringType())) \
.withColumn('numero', f.col('Numero').cast(t.StringType())) \
.withColumn('complemento', f.col('Complemento').cast(t.StringType())) \
.withColumn('bairro', f.col('Bairro').cast(t.StringType())) \
.withColumn('cidade', f.col('Cidade').cast(t.StringType())) \
.withColumn('estado', f.col('Estado').cast(t.StringType())) \
.withColumn('cep', f.col('CEP').cast(t.StringType())) \
.withColumn('telefone1', f.col('Telefone1').cast(t.StringType())) \
.withColumn('telefone2', f.col('Telefone2').cast(t.StringType())) \
.withColumn('telefone3', f.col('Telefone3').cast(t.StringType())) \
.withColumn('site', f.col('Site').cast(t.StringType())) \
.withColumn('email', f.col('Email').cast(t.StringType())) \
.withColumn('data_processamento', f.col('data_processamento').cast(t.IntegerType())) 

### 1.2 Salvar na camada Silver

In [0]:
data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_silver_fii_geral.write \
    .mode('overwrite')\
    .partitionBy("data_processamento") \
    .format('delta')\
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.case_spark_cvm.silver_cvm_fii_geral")